# Ejercicio: tu primer grafo con LangGraph

Vamos a construir, paso a paso, un **agente de soporte IT**: recibe un ticket, decide de qué tipo es y responde con la solución de la base de conocimiento.

```
                    ┌──► [software] ──┐
 ticket ─► [clasificar] ─► [red] ─────┼──► [responder] ─► respuesta
                    ├──► [hardware] ──┤
                    └──► [escalar] ───┘
```

La diferencia con el agente del OpenAI Agents SDK: allí **el modelo decide** qué herramienta usar y en qué orden. Aquí **el flujo lo dibujas tú**: el LLM solo clasifica y redacta, y el camino entre pasos es fijo y visible. Es lo que se busca cuando el proceso tiene que ser predecible y auditable.

**Duración:** unos 30 minutos. Todo se ejecuta en orden; solo hay **dos huecos** para completar, marcados con `# TODO`.

> **Versión resuelta.** Los dos huecos están completos.

### 🧭 Antes de empezar

- **Cómo se usa:** ejecuta las celdas en orden, de arriba abajo, con `Mayús + Enter`. Si una celda da error, lee la última línea: casi siempre dice qué falta.
- **Si algo se queda raro:** menú *Entorno de ejecución → Reiniciar y ejecutar todo*.

**Vocabulario de este notebook**

| Término | Qué es |
|---|---|
| **Estado** | la ficha del ticket que comparten todos los pasos: el texto, la categoría, la solución y la respuesta. |
| **Nodo** | un paso del flujo: una función de Python que lee el estado y devuelve lo que cambia. |
| **Edge** | la flecha entre dos nodos. Un **edge condicional** elige la flecha según el estado. |
| **Compilar** | convertir el dibujo en algo que se puede ejecutar. |

## Paso 0 · Instalación y clave

In [ ]:
!pip install -q --upgrade "langchain>=1.4" langchain-openai "langgraph>=1.2" grandalf

La clave identifica tu cuenta y se factura contra ella. **Nunca** la escribas en el notebook: en Colab, guárdala en *Secretos* (icono de la llave) con el nombre `OPENAI_API_KEY`; en tu ordenador, como variable de entorno. Si no la encuentra, te la pide sin mostrarla.

In [ ]:
import os
from getpass import getpass

# La clave NUNCA se escribe en el notebook. Orden: secretos de Colab -> variable de entorno -> teclado
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
print("Clave cargada:", os.environ["OPENAI_API_KEY"][:7] + "…")

os.environ["LANGSMITH_TRACING"] = "false"   # sin trazas externas: el recorrido lo veremos en el paso 6

from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

llm = ChatOpenAI(model="gpt-6-luna", reasoning={"effort": "low"})
print("Listo.")

## Paso 1 · El estado: la ficha del ticket

Todos los nodos leen y escriben en esta ficha. Cada nodo devuelve **solo los campos que cambia**; LangGraph se encarga de juntar los cambios.

In [ ]:
class Ticket(TypedDict):
    texto: str       # lo que escribe el usuario
    categoria: str   # software | red | hardware | otro
    solucion: str    # lo que dice la base de conocimiento
    respuesta: str   # el mensaje final para el usuario

## Paso 2 · La base de conocimiento

En una empresa sería una wiki o un sistema de tickets. Aquí es un diccionario: tres áreas con sus problemas típicos.

In [ ]:
BASE = {
    "software": {
        "licencia": "Comprueba en el portal de IT que la licencia no ha caducado y reinstala desde el instalador interno.",
        "error":    "Actualiza la aplicación a la última versión y, si sigue fallando, reinstálala.",
        "_otro":    "Reinicia la aplicación y comprueba que tu equipo cumple los requisitos mínimos.",
    },
    "red": {
        "wifi":  "Olvida la red WiFi y vuelve a conectarte. Si eres el único afectado, reinicia el equipo.",
        "vpn":   "Cierra el cliente VPN, vuelve a abrirlo y comprueba que tu contraseña no ha caducado.",
        "_otro": "Pregunta si a tus compañeros les pasa lo mismo y avisa al equipo de redes.",
    },
    "hardware": {
        "lento":     "Cierra lo que no uses y mira en el Administrador de tareas qué consume memoria.",
        "impresora": "Apaga y enciende la impresora y comprueba que no tiene papel atascado.",
        "_otro":     "Reinicia el equipo y revisa los cables. Si sigue igual, abre una incidencia con una foto.",
    },
}


def buscar_en_base(area: str, texto: str) -> str:
    """Devuelve la solución del área cuya palabra clave aparece en el ticket."""
    texto = texto.lower()
    for clave, solucion in BASE[area].items():
        if clave != "_otro" and clave in texto:
            return solucion
    return BASE[area]["_otro"]


print(buscar_en_base("red", "No me conecta la VPN desde casa"))

## Paso 3 · El nodo que clasifica  ✏️ *hueco 1*

Es el único sitio donde el LLM **decide** algo. Escribe las instrucciones de clasificación en lenguaje natural: qué categorías hay y qué debe contestar. El código de debajo se asegura de que la respuesta sea una de las cuatro válidas; si no, la trata como `otro`.

*Pista:* pide que conteste **una sola palabra** y explica brevemente qué va en cada categoría (por ejemplo, la impresora es hardware y la VPN es red).

In [ ]:
CATEGORIAS = ["software", "red", "hardware", "otro"]

INSTRUCCIONES = """Clasifica el ticket de soporte en UNA sola palabra, sin nada más:
- software: aplicaciones, programas, licencias, errores al abrir o instalar algo.
- red: WiFi, internet, VPN, conexión.
- hardware: el equipo físico: ordenador lento, pantalla, teclado, impresora, ruidos.
- otro: cualquier cosa que no sea un problema técnico (nóminas, vacaciones, dudas generales)."""


def clasificar(t: Ticket) -> dict:
    r = llm.invoke(f"{INSTRUCCIONES}\n\nTicket: {t['texto']}")
    categoria = r.text.strip().lower().strip(".")
    if categoria not in CATEGORIAS:
        categoria = "otro"
    return {"categoria": categoria}


# prueba rápida del nodo, sin grafo
clasificar({"texto": "La impresora de la tercera planta no imprime"})

## Paso 4 · Los nodos de cada camino y el que responde

Un nodo por área. Fíjate en que **no llaman al LLM**: buscar en la base es determinista, y así es más barato y no se equivoca. El LLM vuelve a entrar al final, solo para redactar.

In [ ]:
def software(t: Ticket) -> dict:
    return {"solucion": buscar_en_base("software", t["texto"])}

def red(t: Ticket) -> dict:
    return {"solucion": buscar_en_base("red", t["texto"])}

def hardware(t: Ticket) -> dict:
    return {"solucion": buscar_en_base("hardware", t["texto"])}

def escalar(t: Ticket) -> dict:
    return {"solucion": "No es un problema técnico: se deriva a la persona responsable, que contestará en 24 horas."}


def responder(t: Ticket) -> dict:
    r = llm.invoke(
        "Eres el equipo de soporte IT. Redacta una respuesta breve (máximo 3 frases) y amable "
        f"para este ticket: «{t['texto']}». Usa SOLO esta solución, sin añadir pasos: {t['solucion']}"
    )
    return {"respuesta": r.text}

## Paso 5 · Dibujar el grafo  ✏️ *hueco 2*

Primero se añaden los nodos y después las flechas. La flecha que sale de `clasificar` es **condicional**: la función `elegir_camino` mira la categoría y el diccionario dice a qué nodo lleva cada valor.

*Pista:* completa el diccionario con las cuatro categorías: `"software"` va al nodo `"software"`, y así con `red` y `hardware`; `"otro"` va a `"escalar"`.

In [ ]:
g = StateGraph(Ticket)
g.add_node("clasificar", clasificar)
g.add_node("software", software)
g.add_node("red", red)
g.add_node("hardware", hardware)
g.add_node("escalar", escalar)
g.add_node("responder", responder)

g.add_edge(START, "clasificar")


def elegir_camino(t: Ticket) -> str:
    return t["categoria"]


CAMINOS = {"software": "software", "red": "red", "hardware": "hardware", "otro": "escalar"}
g.add_conditional_edges("clasificar", elegir_camino, CAMINOS)

for nodo in ["software", "red", "hardware", "escalar"]:
    g.add_edge(nodo, "responder")
g.add_edge("responder", END)

app = g.compile()
print(app.get_graph().draw_ascii())

## Paso 6 · Probarlo y ver el recorrido

`stream` ejecuta el grafo y va mostrando **cada nodo por el que pasa**: es la traza del agente, sin herramientas externas. La celda comprueba además que cada ticket ha ido por el camino esperado.

In [ ]:
PRUEBAS = [
    ("No puedo instalar el programa de contabilidad, dice que la licencia no es válida.", "software"),
    ("Desde esta mañana no me conecta la VPN.", "red"),
    ("La impresora de la tercera planta no imprime.", "hardware"),
    ("¿Cuándo se cobra la paga extra de diciembre?", "otro"),
]

aciertos = 0
for texto, esperada in PRUEBAS:
    print("\n" + "=" * 60 + f"\nTICKET: {texto}")
    camino = []
    for paso in app.stream({"texto": texto}, stream_mode="updates"):
        nodo, cambios = next(iter(paso.items()))
        camino.append(nodo)
        if nodo == "responder":
            print("RESPUESTA:", cambios["respuesta"])
    print("RECORRIDO:", " → ".join(camino))
    nodo_esperado = "escalar" if esperada == "otro" else esperada
    ok = camino[1] == nodo_esperado
    aciertos += ok
    print("✅ camino correcto" if ok else f"❌ esperaba {esperada}, fue por {camino[1]}")

print(f"\n{aciertos}/{len(PRUEBAS)} tickets por el camino correcto")

## Paso 7 (opcional) · Una persona aprueba antes de responder

Con un **checkpointer** el grafo puede pararse a mitad, guardar la ficha y seguir después. Aquí se para antes de `responder` en **todos** los tickets: una persona revisa la solución propuesta y decide. (Para pararse solo en los escalados, la pausa iría en un nodo propio de esa rama.)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

app_hitl = g.compile(checkpointer=InMemorySaver(), interrupt_before=["responder"])
config = {"configurable": {"thread_id": "ticket-42"}}

app_hitl.invoke({"texto": "Quiero cambiar mis días de vacaciones de agosto."}, config)
pausa = app_hitl.get_state(config)
print("Pausado antes de:", pausa.next)
print("Categoría:", pausa.values["categoria"], "| Solución propuesta:", pausa.values["solucion"])

# la persona aprueba (o cambia la solución con app_hitl.update_state) y el grafo sigue
final = app_hitl.invoke(None, config)
print("\nRESPUESTA ENVIADA:", final["respuesta"])

## Paso 8 (opcional) · Para seguir practicando

- Añade una quinta categoría, `accesos` (contraseñas, cuentas bloqueadas), con su entrada en `BASE`, su nodo y su camino.
- Cambia la prueba del paso 6 para incluir un ticket ambiguo («el Teams va lentísimo») y mira por dónde va. ¿Lo arreglarías en las instrucciones o en el código?

## Resumen

| | OpenAI Agents SDK (S4_Lab_Agents_SDK) | LangGraph (este ejercicio) |
|---|---|---|
| ¿Quién decide el siguiente paso? | El modelo | Tú, con el grafo |
| Cuándo encaja | Conversaciones abiertas, muchas herramientas | Procesos con pasos fijos que hay que poder auditar |
| Cómo se ve lo que pasó | Trazas en el panel de OpenAI | El recorrido de nodos (y LangSmith, si lo activas) |